In [1]:
import matplotlib.pyplot as plt
from matplotlib import markers
import pandas as pd
import numpy as np
import os

# Ahora importa el módulo
# from RRAM.Representate import config_ax_IV, setup_paper_plt
    
# setup_paper_plt(plt, latex=True, scaling=3)

In [2]:
def split_dataframe_at_max(df):
    # Encontrar el índice del valor máximo de voltaje
    max_index = abs(df["voltage"]).idxmax()

    # Dividir el DataFrame
    df_ascendente = df.iloc[: max_index + 1]  # Hasta el máximo (inclusive)
    df_descendente = df.iloc[max_index+1:]  # Desde el máximo hasta el final

    return df_ascendente, df_descendente

In [3]:
def leer_datos_exp(num_curva: int):
    ruta_set = (os.getcwd()+ f"/Datos_Experimentales/Medidas_Experimentales_RRAM/Cycle_p_{num_curva}.txt")
    ruta_reset = (os.getcwd()+ f"/Datos_Experimentales/Medidas_Experimentales_RRAM/Cycle_n_{num_curva}.txt")
    
    # Definir estructura de datos común
    dtype = [("voltage", "f8"), ("current", "f8"), ("time", "f8")]

    # Leer archivos con numpy
    datos_set = np.genfromtxt(ruta_set, dtype=dtype, encoding="utf-8")
    datos_reset = np.genfromtxt(ruta_reset, dtype=dtype, encoding="utf-8")

    # Convertir a DataFrames
    df_set = pd.DataFrame(datos_set)
    df_reset = pd.DataFrame(datos_reset)
    
    df_pp_set, df_sp_set = split_dataframe_at_max(df_set)
    df_pp_reset, df_sp_reset = split_dataframe_at_max(df_reset)

    return df_pp_set, df_sp_set, df_pp_reset, df_sp_reset

In [4]:
def leer_datos_sim(num_simulacion: int):
    """
    Lee los datos de simulación de archivos CSV y los organiza en DataFrames.
    Esta función toma un número de simulación como entrada, construye las rutas
    de los archivos correspondientes a los datos de "set" y "reset", y lee los
    datos de estos archivos en formato CSV. Los datos se convierten en DataFrames
    con columnas reordenadas.
    Args:
        num_simulacion (int): Número de la simulación para identificar las rutas
                              de los archivos de datos.
    Returns:
        Tuple[pd.DataFrame, pd.DataFrame]: Una tupla que contiene dos DataFrames:
            - df_set: DataFrame con los datos de "set" (columnas: "voltage", "current", "time").
            - df_reset: DataFrame con los datos de "reset" (columnas: "voltage", "current", "time").
    Notas:
        - Los archivos CSV deben estar ubicados en las rutas esperadas dentro del
          directorio de trabajo actual.
        - Solo se leen las primeras tres columnas de los archivos CSV.
        - Los nombres de las columnas esperadas son "time", "voltage" y "current".
        - Se omite la primera fila de los archivos CSV al leer los datos.
    """
    
    ruta_pp_set = (
        os.getcwd()
        + f"/Results/simulation_{num_simulacion}/set/resultados_pp_set_{num_simulacion}.csv"
    )
    ruta_sp_set = (
        os.getcwd()
        + f"/Results/simulation_{num_simulacion}/set/resultados_sp_set_{num_simulacion}.csv"
    )
    
    # Leer solo las primeras 3 columnas
    usecols = (0, 1, 2)  # Índices de las columnas que queremos leer

    # Definir tipos de datos para cada columna
    dtypes = {"time": "float64", "voltage": "float64", "current": "float64"}
    
    # Leer archivos con numpy
    datos_pp_set = pd.read_csv(
        ruta_pp_set,
        encoding="utf-8",
        usecols=usecols,
        names=["time", "voltage", "current"],
        dtype=dtypes,
        skiprows=1,
    )
    datos_sp_set = pd.read_csv(
        ruta_sp_set,
        encoding="utf-8",
        usecols=usecols,
        names=["time", "voltage", "current"],
        dtype=dtypes,
        skiprows=1,
    )

    # Convertir a DataFrames
    df_pp_set = pd.DataFrame(datos_pp_set)
    df_sp_set = pd.DataFrame(datos_sp_set)
    
    df_pp_set = df_pp_set.reindex(columns=["voltage", "current", "time"])
    df_sp_set = df_sp_set.reindex(columns=["voltage", "current", "time"])

    return df_pp_set, df_sp_set

In [5]:
def leer_npz_a_dataframe(ruta_archivo):
    """
    Lee un archivo .npz y lo convierte en un DataFrame de pandas.

    Parámetros:
        ruta_archivo (str): Ruta al archivo .npz.

    Retorna:
        pd.DataFrame: DataFrame con los datos del archivo .npz.
    """
    # Cargar el archivo .npz
    datos_npz = np.load(ruta_archivo)

    # Extraer los datos y el encabezado
    datos = datos_npz["datos"]  # Matriz de datos (7857, 3)
    header = datos_npz["header"][0]  # Encabezado como string

    # Dividir el encabezado en nombres de columnas
    columnas = header.split(",")  # Separar por comas

    # Crear el DataFrame
    df = pd.DataFrame(datos, columns=columnas)

    return df


In [6]:
def comparar_intensidades(df_exp, df_sim):
    """
    Compara intensidades experimentales con simuladas buscando para cada voltaje experimental
    el voltaje simulado más cercano y calculando la diferencia y valor absoluto entre intensidades.

    Parámetros:
    - df_exp: DataFrame experimental con columnas ['voltaje', 'intensidad', 'tiempo']
    - df_sim: DataFrame simulado con columnas ['voltaje', 'intensidad', 'tiempo']

    Retorna:
    - DataFrame con columnas ['voltaje_exp', 'intensidad_exp', 'voltaje_sim_cercano', 'intensidad_sim', 'diferencia', 'abs_diferencia']
    """
    voltajes_sim = df_sim["Voltaje [V]"].values
    intensidades_sim = df_sim["Intensidad [A]"].values

    resultados = []

    for _, row_exp in df_exp.iterrows():
        v_exp = row_exp["voltage"]
        i_exp = row_exp["current"]

        # índice del voltaje simulado más cercano
        idx_min = np.argmin(np.abs(voltajes_sim - v_exp))
        v_sim_cercano = voltajes_sim[idx_min]
        i_sim_cercano = intensidades_sim[idx_min]

        diff = i_exp - i_sim_cercano
        abs_diff = abs(diff)

        resultados.append(
            {
                "voltaje_exp": v_exp,
                "intensidad_exp": i_exp,
                "voltaje_sim_cercano": v_sim_cercano,
                "intensidad_sim": i_sim_cercano,
                "diferencia": diff,
                "abs_diferencia": abs_diff,
            }
        )

    return pd.DataFrame(resultados)

In [7]:
errores_totales = []

# Datos de la curva experimental que uso en la simulación
(
    df_pp_set_exp,
    df_sp_set_exp,
    df_pp_reset_exp,
    df_sp_reset_exp,
) = leer_datos_exp(num_curva=1000)

print("df_pp_set_exp:" , df_pp_set_exp)

for num_sim in range(1, 100):
    
    data_path = os.getcwd() + f"/Results/simulation_{num_sim}/"
    
    df_pp_set_sim = leer_npz_a_dataframe(data_path + f"Data_pp_set_{num_sim}.npz")
    df_sp_set_sim = leer_npz_a_dataframe(data_path + f"Data_sp_set_{num_sim}.npz")
    df_pp_reset_sim = leer_npz_a_dataframe(data_path + f"Data_pp_reset_{num_sim}.npz")
    df_sp_set_sim = leer_npz_a_dataframe(data_path + f"Data_sp_reset_{num_sim}.npz")

    df_pp_set_comparacion = comparar_intensidades(df_pp_set_exp, df_pp_set_sim)
    df_sp_set_comparacion = comparar_intensidades(df_sp_set_exp, df_sp_set_sim)
    df_pp_reset_comparacion = comparar_intensidades(df_pp_reset_exp, df_sp_set_sim)
    df_sp_reset_comparacion = comparar_intensidades(df_sp_reset_exp, df_sp_set_sim)
    
    error_pp_set = df_pp_set_comparacion["abs_diferencia"].sum() # Convertir a mA
    error_sp_set = df_sp_set_comparacion["abs_diferencia"].sum() # Convertir a mA
    error_pp_reset = df_pp_reset_comparacion["abs_diferencia"].sum() # Convertir a mA
    error_sp_reset = df_sp_reset_comparacion["abs_diferencia"].sum() # Convertir a mA
    
    print(
        f"Curva {num_sim}: Error PP set = {error_pp_set:.4}, Error SP set = {error_sp_set:.4}, Error PP reset = {error_pp_reset:.4}, Error SP reset = {error_sp_reset:.4}"
    )
    
    # Almacenar la suma de errores
    error_total = error_pp_set + error_sp_set + error_pp_reset + error_sp_reset
    errores_totales.append(error_total)
    
# Crear lista de tuplas (número_curva, error)
errores_con_indices = [(i + 2, error) for i, error in enumerate(errores_totales)]

# Ordenar por error (menor a mayor)
errores_ordenados = sorted(errores_con_indices, key=lambda x: x[1])

# Obtener las 5 mejores curvas
mejores_curvas = errores_ordenados[:10]

print("\nLas 10 mejores curvas son:")
for num_sim, error in mejores_curvas:
    print(f"Simulación {num_sim}: Error total = {error:.6f}")

df_pp_set_exp:      voltage   current      time
0       0.01  0.000019  0.065563
1       0.02  0.000037  0.113130
2       0.03  0.000055  0.161520
3       0.04  0.000074  0.209118
4       0.05  0.000093  0.257350
..       ...       ...       ...
105     1.06  0.024180  3.437529
106     1.07  0.024510  3.453731
107     1.08  0.024850  3.469242
108     1.09  0.025260  3.485114
109     1.10  0.025655  3.501357

[110 rows x 3 columns]
Curva 1: Error PP set = 0.1342, Error SP set = 1.438, Error PP reset = 2.033, Error SP reset = 0.6296
Curva 2: Error PP set = 0.07833, Error SP set = 1.438, Error PP reset = 2.003, Error SP reset = 0.6004
Curva 3: Error PP set = 0.1475, Error SP set = 1.438, Error PP reset = 1.921, Error SP reset = 0.5311
Curva 4: Error PP set = 0.1584, Error SP set = 1.438, Error PP reset = 1.919, Error SP reset = 0.5281
Curva 5: Error PP set = 0.1288, Error SP set = 1.438, Error PP reset = 2.001, Error SP reset = 0.599
Curva 6: Error PP set = 0.113, Error SP set = 1.438, Er

In [8]:
# def plot_all_IV_best_ajust(
#     df_exp,
#     df_sim,
#     num_curva: int
# ):

#     # Configuración de la figura
#     fig, axes = plt.subplots(figsize=(12, 9))
#     setup_paper_plt(plt, latex=True, scaling=2.5)
#     config_ax_IV(axes)

#     axes.set_xlabel("Voltage (V)")
#     axes.set_ylabel("Current (A)")
#     axes.set_yscale("log")

#     # Scatter de SET y RESET
#     axes.scatter(
#         df_sim["voltage"],
#         abs(df_sim["current"]),
#         color="red",
#         s=15,
#         marker=markers.MarkerStyle("o"),
#         facecolors="white",
#         label="SET simulado",
#     )

#     # Curvas experimentales
#     axes.plot(
#         df_exp["voltage"],
#         abs(df_exp["current"]),
#         "black",
#         linewidth=2.5,
#         label="Num. Curva: " + str(num_curva),
#     )

#     # Leyenda ajustada en la parte inferior izquierda
#     axes.legend(
#         labelspacing=0.3,
#         handletextpad=0.2,
#         handlelength=1.0,
#         borderaxespad=0.2,
#         loc="lower right",
#     )

In [9]:
# # Uno los dataframes de la mejor curva
# df_set_sim = pd.concat([df_pp_set, df_sp_set], ignore_index=True)

# fig, ejes = plt.subplots(figsize=(12, 9))
# setup_paper_plt(plt, latex=True, scaling=2.5)
# config_ax_IV(ejes)

# # Scatter de SET y RESET
# ejes.scatter(
#     df_set_sim["voltage"],
#     abs(df_set_sim["current"]),
#     color="red",
#     s=35,
#     marker=markers.MarkerStyle("o"),
#     facecolors="white",
#     label="SET simulado",
# )
    
# # Configuración de la figura
# ejes.set_xlabel("Voltage (V)")
# ejes.set_ylabel("Current (A)")
# ejes.set_yscale("log")

# for i in [1497, 1510, 1495, 1493, 1496]:
#     df_pp_set_exp, df_sp_set_exp = leer_datos_exp(num_curva = i)
#     df_set_exp = pd.concat([df_pp_set_exp, df_sp_set_exp], ignore_index=True)
    
#     plot_all_IV_best_ajust(
#         df_exp = df_set_exp,
#         df_sim= df_set_sim,
#         num_curva= i,)
        
#         # Curvas experimentales
#     ejes.plot(
#         df_set_exp["voltage"],
#         abs(df_set_exp["current"]),
#         linewidth=2.5,
#         label=f"Exp. Curva {num_curva}",
#     )

#     # Leyenda ajustada en la parte inferior izquierda
#     ejes.legend(
#         labelspacing=0.3,
#         handletextpad=0.2,
#         handlelength=1.0,
#         borderaxespad=0.2,
#         loc="lower right",
#     )

# # Crear figura con subplots (3x2 para tener espacio para 5 gráficas)
# fig, axes = plt.subplots(2, 3, figsize=(25, 15))
# setup_paper_plt(plt, latex=True, scaling=1.5)

# axes = axes.flatten()  # Convertir matriz 2D de axes a 1D para fácil iteración

# # Iterar sobre las curvas seleccionadas
# for idx, num_curva in enumerate([1497, 1510, 1495, 1493, 1496]):
#     # Leer datos
#     df_pp_set_exp, df_sp_set_exp = leer_datos_exp(num_curva=num_curva)
#     df_set_exp = pd.concat([df_pp_set_exp, df_sp_set_exp], ignore_index=True)

#     # Configurar cada subplot
#     ax = axes[idx]
#     ax.set_yscale("log")
#     ax.set_xlabel("Voltage (V)")
#     ax.set_ylabel("Current (A)")

#     # Scatter de datos simulados
#     ax.scatter(
#         df_set_sim["voltage"],
#         abs(df_set_sim["current"]),
#         color="red",
#         s=15,
#         marker=markers.MarkerStyle("o"),
#         facecolors="white",
#         label="SET simulado",
#     )

#     # Plot datos experimentales
#     ax.plot(
#         df_set_exp["voltage"],
#         abs(df_set_exp["current"]),
#         "black",
#         linewidth=2.5,
#         label=f"Exp. Curva {num_curva}",
#     )

#     ax.legend(
#         labelspacing=0.3,
#         handletextpad=0.2,
#         handlelength=1.0,
#         borderaxespad=0.2,
#         loc="lower right",
#     )
#     ax.set_title(f"Curva {num_curva}")

# # Eliminar el último subplot vacío
# axes[-1].remove()

# # Ajustar el espaciado entre subplots
# plt.tight_layout()
# plt.show()